# Logistic Regression with a Hidden Layer

## Entropy and Cross-Entropy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p0 = np.array([1])

p1 = np.array([1/2, 1/2])

p2 = np.array([1/100, 99/100])

p3 = np.array([1/3, 1/3, 1/3])

H_p0 = - np.sum(p0 * np.log(p0))
print(f"H_p0: {H_p0:.4f}")

H_p1 = - np.sum(p1 * np.log(p1))
print(f"H_p1: {H_p1:.4f}")

H_p2 = - np.sum(p2 * np.log(p2))
print(f"H_p2: {H_p2:.4f}")

H_p3 = - np.sum(p3 * np.log(p3))
print(f"H_p3: {H_p3:.4f}")

$$
L = \text{CE} = -\frac{1}{n}\sum_{i=1}^n \sum_{k=1}^K y_{ik} \log \hat{y}_{ik}
$$

## Why Cross-Entropy Loss instead of MSE?

## Binary Classification

### Logistic Regression Model

$$
\sigma(o) = \frac{1}{1 + e^{-o}}
$$

## The Logistic Regression as an ANN

### Binary Cross-Entropy Loss

$$
L = \text{BCE}(y,\hat{y}) = - \frac{1}{n}\sum_{i=1}^n \left( y_i \log \hat{y}_i + (1 - y_i) \log (1 - \hat{y}_i) \right)
$$

### Logistic regression has a linear decision boundary

## Logistic Regression with a Hidden Layer

$$
\begin{align*}
H & = X W_h + b_h \\
Z & = \varphi(H) \\
O & = Z W_o + b_o \\
\hat{y} & = \sigma(O) \\
L & = -\frac{1}{n} \sum_{i=1}^n \left( y_i \log \hat{y}_i + (1 - y_i) \log (1 - \hat{y}_i) \right)
\end{align*}
$$

### Gradients

$$
\begin{align*}
\frac{\partial L_i}{\partial o_i}
& = -\left( y_i \frac{\sigma(o_i)(1 - \sigma(o_i))}{\sigma(o_i)} - (1 - y_i) \frac{\sigma(o_i)(1 - \sigma(o_i))}{1 - \sigma(o_i)} \right) \\
& = -\left( y_i (1 - \hat{y}_i) - (1 - y_i) \hat{y}_i \right) \\
& = -(y_i - \hat{y}_i)
\end{align*}
$$

$$
\underbrace{\frac{\partial L}{\partial W_o}}_{k \times 1} = -\frac{1}{n} Z^\top e
\qquad
\underbrace{\frac{\partial L}{\partial b_o}}_{1 \times 1} = -\frac{1}{n} \mathbf{1}^\top e
$$

$$
\underbrace{\frac{\partial L}{\partial H}}_{n \times k} = -\frac{1}{n} \left( e \, W_o^\top \right) \odot \varphi'(H)
\qquad
\underbrace{\frac{\partial L}{\partial W_h}}_{p \times k} = -\frac{1}{n} X^\top \left( \left( e \, W_o^\top \right) \odot \varphi'(H) \right)
\qquad
\underbrace{\frac{\partial L}{\partial b_h}}_{1 \times k} = -\frac{1}{n} \mathbf{1}^\top \left( \left( e \, W_o^\top \right) \odot \varphi'(H) \right)
$$

### A small numerical example

## Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression

X_moons, y_moons = make_moons(n_samples=300, noise=0.25, random_state=0)
y_moons = y_moons.reshape(-1, 1).astype(float)           # n x 1

plt.figure(figsize=(6, 5))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons.ravel(), cmap='bwr', edgecolor='k', s=50)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Two half-moons')
plt.grid(True, alpha=0.3)
plt.show()

### The forward pass and the gradients in NumPy

In [ ]:
def sigmoid(O):
    return 1 / (1 + np.exp(-O))

def relu(H):
    return np.maximum(0, H)

def relu_prime(H):
    return (H > 0).astype(float)

def tanh_prime(H):
    return 1 - np.tanh(H) ** 2

def forward(X, params, activation=relu):
    W_h, b_h, W_o, b_o = params
    H = X @ W_h + b_h
    Z = activation(H)
    return H, Z, sigmoid(Z @ W_o + b_o)

def binary_cross_entropy(y, y_hat):
    return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))

def gradients(X, y, params, activation=relu, activation_prime=relu_prime):
    W_h, b_h, W_o, b_o = params
    n = X.shape[0]

    H, Z, y_hat = forward(X, params, activation)
    error = y - y_hat

    dW_o = -(Z.T @ error) / n
    db_o = -error.sum(axis=0, keepdims=True) / n

    Delta = (error @ W_o.T) * activation_prime(H)
    dW_h = -(X.T @ Delta) / n
    db_h = -Delta.sum(axis=0, keepdims=True) / n

    return dW_h, db_h, dW_o, db_o

# The numerical example from above
X_small = np.array([[4.0], [2.0]])
y_small = np.array([[1.0], [0.0]])
params_small = (np.array([[0.25, 0.5]]), np.array([[0.1, -1.1]]),
                np.array([[0.8], [-1.0]]), np.array([[-0.1]]))

_, _, y_hat_small = forward(X_small, params_small)
print("y_hat:", y_hat_small.ravel().round(4))
print("Loss:", round(binary_cross_entropy(y_small, y_hat_small), 4))

for name, grad in zip(["dW_h", "db_h", "dW_o", "db_o"], gradients(X_small, y_small, params_small)):
    print(f"{name}:", grad.round(4).tolist())

### Checking the gradients against PyTorch

In [ ]:
def init_params(n_features, n_hidden, seed=0):
    rng = np.random.default_rng(seed)
    return [
        rng.normal(0, 0.5, (n_features, n_hidden)),   # W_h
        np.zeros((1, n_hidden)),                      # b_h
        rng.normal(0, 0.5, (n_hidden, 1)),            # W_o
        np.zeros((1, 1)),                             # b_o
    ]

params_moons = init_params(n_features=2, n_hidden=8)

activations = {
    "relu": (relu, relu_prime, torch.relu),
    "tanh": (np.tanh, tanh_prime, torch.tanh),
}

X_double = torch.tensor(X_moons)                       # float64 for an accurate comparison
y_double = torch.tensor(y_moons)

for name, (act, act_prime, torch_act) in activations.items():
    grads_numpy = gradients(X_moons, y_moons, params_moons, act, act_prime)

    params_torch = [torch.tensor(p, requires_grad=True) for p in params_moons]
    W_h_t, b_h_t, W_o_t, b_o_t = params_torch
    y_hat_t = torch.sigmoid(torch_act(X_double @ W_h_t + b_h_t) @ W_o_t + b_o_t)
    loss_t = nn.functional.binary_cross_entropy(y_hat_t, y_double)   # mean over observations
    loss_t.backward()

    match = all(np.allclose(g, p.grad.numpy()) for g, p in zip(grads_numpy, params_torch))
    print(f"{name}: gradients match PyTorch autograd: {match}")
    assert match

### Training

In [ ]:
def train(X, y, params, activation, activation_prime, learning_rate=0.5, epochs=5000):
    params = [p.copy() for p in params]
    losses = []
    for epoch in range(epochs):
        grads = gradients(X, y, params, activation, activation_prime)
        for param, grad in zip(params, grads):
            param -= learning_rate * grad              # theta <- theta - eta * dL/dtheta
        losses.append(binary_cross_entropy(y, forward(X, params, activation)[2]))
    return params, losses

trained, losses = train(X_moons, y_moons, params_moons, np.tanh, tanh_prime)
y_hat_nn = forward(X_moons, trained, np.tanh)[2]
accuracy_nn = ((y_hat_nn > 0.5) == y_moons).mean()

logistic = LogisticRegression().fit(X_moons, y_moons.ravel())
accuracy_logistic = logistic.score(X_moons, y_moons.ravel())

print(f"Logistic regression (no hidden layer): training accuracy {accuracy_logistic:.3f}")
print(f"Network with a hidden layer (tanh):    training accuracy {accuracy_nn:.3f}, loss {losses[-1]:.4f}")

# Decision regions
xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(losses)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary cross-entropy')
axes[0].set_title('Training loss')
axes[0].grid(True, alpha=0.3)

panels = [("Logistic regression", logistic.predict_proba(grid)[:, 1]),
          ("Hidden layer (tanh)", forward(grid, trained, np.tanh)[2].ravel())]

for ax, (title, probs) in zip(axes[1:], panels):
    contour = ax.contourf(xx, yy, probs.reshape(xx.shape), levels=20, cmap='RdYlBu_r', alpha=0.6)
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons.ravel(), cmap='bwr', edgecolor='k', s=30)
    ax.set_title(title)
    fig.colorbar(contour, ax=ax, label='P(class 1)')

plt.tight_layout()
plt.show()

### The same model in PyTorch

In [ ]:
torch.manual_seed(0)

X_tensor = torch.tensor(X_moons, dtype=torch.float32)
y_tensor = torch.tensor(y_moons, dtype=torch.float32)

model = nn.Sequential(
    nn.Linear(2, 8),     # X W_h + b_h  (the weight is stored as 8 x 2, the transpose of W_h)
    nn.Tanh(),           # Z = tanh(H)
    nn.Linear(8, 1),     # logits O = Z W_o + b_o
)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(2000):
    loss = criterion(model(X_tensor), y_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    probs = torch.sigmoid(model(X_tensor))
    accuracy = ((probs > 0.5).float() == y_tensor).float().mean()

print(f"Final loss: {loss.item():.4f}, training accuracy: {accuracy.item():.3f}")